In [ ]:
import os
os.chdir("/orcd/archive/abugoot/001/Projects/paolo/tde_main/")

import hydra
import argparse
from omegaconf import DictConfig, OmegaConf
import matplotlib.pyplot as plt
import numpy as np
import torch
import logging
from utils.seed import seed_everything  # Import seeding utility

# Set random seed for reproducibility
RANDOM_SEED = 42
seed_everything(RANDOM_SEED, deterministic=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(device)

In [ ]:
def load_cfg_and_ckpt(ckpt_dir):
    # Resolve and validate checkpoint directory
    experiment_dir = os.path.abspath(os.path.expanduser(str(ckpt_dir)))
    cfg_path = os.path.join(experiment_dir, 'config.yaml')
    ckpt_path = os.path.join(experiment_dir, 'best_model.pt')
    # Load trained config and use it as the active config
    cfg = OmegaConf.load(cfg_path)
    return cfg, ckpt_path

def load_models(cfg,ckpt_path):
    encoder = hydra.utils.instantiate(cfg.encoder).to(device)
    generator = hydra.utils.instantiate(cfg.generator).to(device)

    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

    encoder.load_state_dict(checkpoint['encoder_state_dict'])
    generator.load_state_dict(checkpoint['generator_state_dict'])
    encoder.eval()
    generator.eval()

    return encoder, generator

def plot_letters(cfg, encoder, generator, eval=True, font_offset=0, num_letters=6):
    dataset = hydra.utils.instantiate(cfg.dataset, eval=eval)

    letters = dataset.eval_letters if eval else dataset.train_letters

    example_indices = ((dataset.num_fonts - 1)) * np.arange(min(len(letters),num_letters)) + font_offset

    num_examples = len(example_indices)

    num_rows = 3
    fig_height = 12
    fig, axes = plt.subplots(num_rows, num_examples, figsize=(4*num_examples, fig_height))
    if num_examples == 1:
        axes = axes.reshape(num_rows, 1)
        
    num_sets = 10
    for i, idx in enumerate(example_indices):
        #print(i)
        # Determine which letter and fonts we're looking at
        letter = dataset.train_letters[idx // (dataset.num_fonts - 1)] if not eval else dataset.eval_letters[idx // (dataset.num_fonts - 1)]
        font = idx % (dataset.num_fonts - 1)
        
        # Get dataset item
        source_samples_collection = []
        target_samples_collection = []
        gen_samples_collection = []
        for _ in range(num_sets):
            
            #print("!!!!!!", i, _)
            item = dataset[idx]
            source_samples = item['source_samples']
            target_samples = item['target_samples']
        
            source_samples_collection.append(source_samples)
            target_samples_collection.append(target_samples)
            
            with torch.no_grad():
                # Prepare tensors
                src_tensor = source_samples.to(device)
                tgt_tensor = target_samples.to(device)
                # Encode to latents (batch with single set)
                src_latent = encoder(src_tensor.unsqueeze(0))  # [1, latent_dim]
                #print(cfg.model.get('source_only'))
                if cfg.model.get('source_only', False):
                    tgt_latent = None 
                else:
                    tgt_latent = encoder(tgt_tensor.unsqueeze(0))  # [1, latent_dim]
                # Generate from source samples conditioned on src/target latents
                gen = generator.sample(src_tensor, src_latent, tgt_latent)  # [1, set_size, 2]
                gen_np = gen.squeeze(0).detach().cpu()
            
            gen_samples_collection.append(gen_np)
        
        source_samples = torch.cat(source_samples_collection, dim=0)
        target_samples = torch.cat(target_samples_collection, dim=0)
        gen_samples = torch.cat(gen_samples_collection, dim=0)

        # Plot source samples
        axes[0, i].scatter(source_samples[:, 0], source_samples[:, 1], 
                        alpha=0.6, s=1, c='blue')
        axes[0, i].set_title(f"Source: Letter '{letter}' (Font {font})")
        axes[0, i].set_xlim(0, 1)
        axes[0, i].set_ylim(0, 1)
        axes[0, i].set_aspect('equal')
        axes[0, i].grid(True, alpha=0.3)
        axes[0, i].invert_yaxis()  # Invert y-axis to match image coordinates
        
        # Plot target samples  
        axes[1, i].scatter(target_samples[:, 0], target_samples[:, 1], 
                        alpha=0.6, s=1, c='red')
        axes[1, i].set_title(f"Target: Letter '{letter}' (Font {font+1})")
        axes[1, i].set_xlim(0, 1)
        axes[1, i].set_ylim(0, 1)
        axes[1, i].set_aspect('equal')
        axes[1, i].grid(True, alpha=0.3)
        axes[1, i].invert_yaxis()  # Invert y-axis to match image coordinates
        
        axes[2, i].scatter(gen_samples[:, 0], gen_samples[:, 1], alpha=0.6, s=1, c='green')
        axes[2, i].set_title(f"Generated from '{letter}' (Font {font}→{font+1})")
        axes[2, i].set_xlim(0, 1)
        axes[2, i].set_ylim(0, 1)
        axes[2, i].set_aspect('equal')
        axes[2, i].grid(True, alpha=0.3)
        axes[2, i].invert_yaxis()

    plt.tight_layout() 

    plt.show()


In [ ]:
# NOTE: THIS IS OUR METHOD (TDEs)

ckpt_dir = "./outputs/letters_coupled_exp_de2f68fc98974210d2d4d720ec6e42b5"

cfg, ckpt_path = load_cfg_and_ckpt(ckpt_dir)
encoder, generator = load_models(cfg, ckpt_path)


In [ ]:
# Training data
plot_letters(cfg, encoder, generator, eval=False, font_offset=5, num_letters=10)

In [ ]:
# Evaluation data
plot_letters(cfg, encoder, generator, eval=True, font_offset=8, num_letters=3)